In [ ]:
%reload_ext autoreload
%autoreload 2
from importlib import reload

import os
import sys
import pickle
#import dill
import logging
import warnings
import numpy as np
import astropy as ap
import scipy as sp
import scipy.stats
import matplotlib as mpl
import matplotlib.pyplot as plt

import h5py
import tqdm.notebook as tqdm

import kalepy as kale
import kalepy.utils
import kalepy.plot

import holodeck as holo
import holodeck.sams
import holodeck.gravwaves
from holodeck import cosmo, utils, plot, discrete, sams, host_relations, _PATH_DATA
from holodeck.constants import MSOL, PC, YR, MPC, GYR, SPLC
from pathlib import Path

import compare_discrete

# Silence annoying numpy errors
np.seterr(divide='ignore', invalid='ignore', over='ignore')
warnings.filterwarnings("ignore", category=UserWarning)

# Plotting settings
mpl.rc('font', **{'family': 'serif', 'sans-serif': ['Times'], 'size': 15})
mpl.rc('lines', solid_capstyle='round')
mpl.rc('mathtext', fontset='cm')
plt.rcParams.update({'grid.alpha': 0.5})
mpl.style.use('default')   # avoid dark backgrounds from dark theme vscode

log = holo.log
log.setLevel(logging.INFO)

# ---- Define filepath containing simulation galaxy merger data files ----#
# ---- (if using files not in _PATH_DATA) ---- #
_HOME_PATH = Path('~/').expanduser()
p = os.path.join(_HOME_PATH, 'cosmo_sim_merger_data')
if os.path.exists(p):
    _SIM_MERGER_PATH = p
else:
    p = os.path.join(_HOME_PATH, 'nanograv/cosmo_sim_merger_data')
    if os.path.exists(p):
        _SIM_MERGER_PATH = p
    else:
        _SIM_MERGER_PATH = _PATH_DATA
print(f"{_SIM_MERGER_PATH=}")
# ------------------------------------------------------------------------ #


# ---- EDIT PARAMS AS NEEDED TO MATCH input files
NFREQS = 40 
NLOUD = 5
NREALS = 500
#TAU = 1.0 # fixed hardening timescale [Gyr]
#TAU = 2.0 # fixed hardening timescale [Gyr]

In [ ]:
print(_PATH_DATA)

In [ ]:
def gwb_amps(sams=None, dpops=None, fname='compare_gwb_amps', color_hardmods=False, gpf_flags=None):
    
    fig, axs = plt.subplots(nrows=3, ncols=1, sharex=True, figsize=[10,10])

    freqs_to_plot = np.array([1/(10*YR), 1/(3*YR), 1/YR])
    
    for i,f in enumerate(freqs_to_plot):
 
        if i==freqs_to_plot.size-1:
            axs[i].set(xlabel=r'$\log_{10}(A_\mathrm{yr})$')
        axs[i].set(ylabel='Probability Density')
        axs[i].grid(alpha=0.2)
        
        if sams is not None:
            for k,s in enumerate(sams):
                freqs_sam = s.PARS['freqs']
                idx_sam = np.where(np.abs(freqs_sam-f)==np.abs(freqs_sam-f).min())[0]
                hctot = calc_hctot(s.gwb_sam)
                amp_sam = hctot[idx_sam,:].flatten()
                
                lw = 1.5
                col = 'k'
                ls = '-'
                if s.model_type == 'old':
                    lw=1.5
                    col='k'
                    ls='-'
                elif s.model_type == 'old_rc100':
                    lw=2.5
                    col='k'
                    ls=':' if dpops is None else '-'
                elif s.model_type == 'ph15':
                    lw=2.5 if dpops is None else 2
                    col='darkred' if dpops is None else 'k'
                    ls='-.'
                elif s.model_type == 'old_2s':
                    lw=2
                    col='k'
                    ls='--'
                elif s.model_type == 'astr_rc100':
                    lw=2.5 if dpops is None else 3
                    col='r' if dpops is None else 'k'
                    ls='-'
                elif s.model_type == 'astr_nuo0':
                    lw=2
                    col='darkgray'
                    ls='-.'
                elif s.model_type == 'astr':
                    lw=2
                    col='darkgray'
                    ls='-' if dpops is None else ':'
                else:
                    raise ValueError(f"havent defined plot style for {s.model_type=}")

                lbl = s.model_type
                gpf_lbls = [' (GMR)', ' (GPF)']
                if len(gpf_flags) == len(sams):
                    lbl += gpf_lbls[gpf_flags[k]] 
                    
                kale.dist1d(np.log10(amp_sam), density=True, hist=False, confidence=False, carpet=False, 
                            lw=lw, ls=ls, color=col, label=lbl, ax=axs[i])

        if dpops is not None:
            for d in dpops: 
                if sams is not None:
                    if np.any(d.freqs!=freqs_sam):
                        print(f"WARNING: SAM and sim have different frequency arrays.")
                        idx_sim = np.where(np.abs(d.freqs-f)==np.abs(d.freqs-f).min())[0]
                    else:
                        idx_sim = idx_sam
                else:
                    idx_sim = np.where(np.abs(d.freqs-f)==np.abs(d.freqs-f).min())[0]

                amp_sim = d.gwb.strain[idx_sim,:].flatten()  #`strain` is equivalent to `both`
        
                if np.any(~np.isfinite(amp_sim)):
                    print(f"skipping {d.lbl} ({amp_sim.min()}, {amp_sim.max()})")
                else:
                    #print(f"{d.lbl} ({amp_sim.min()}, {amp_sim.max()})")
                    #ls='--' if 'bh0' in d.lbl else '-'
                    print(f"{d.hard_model_type=}")
                    if color_hardmods:
                        col='g' if d.hard_model_type=='old_rc100' else d.color
                        col='k' if d.hard_model_type=='old_rc10' else d.color
                        col='b' if d.hard_model_type=='ph15' else d.color
                        col='m' if d.hard_model_type=='ph15_rc10' else d.color
                    elif not color_hardmods and 'bh0' in d.lbl:
                        col = 'c' 
                    else: 
                        col = d.color
                    #ls = '-' if d.hard_model_type=='old_rc100' else ':'
                    ls = '-' if d.hard_model_type=='ph15' else ':'
                    if 'rTNG300' in d.lbl:
                        ls = '--'
                    else:
                        ls = '-' if d.hard_model_type=='ph15' else ':'
                    kale.dist1d(np.log10(amp_sim), density=True, hist=False, confidence=False, 
                                carpet=False, label='_'.join((d.lbl,d.hard_model_type)), lw=d.lw, 
                                ls=ls, color=col, alpha=0.5, ax=axs[i])
                
        axs[i].set(title=f"Comparison of GWB amplitudes at f={f*YR:.1g} [1/YR]")
        axs[i].legend(fontsize=9)

    plt.suptitle(fname)
    fig.savefig(f'{fname}_nloud{NLOUD}_nreals{NREALS}.png')
    #plt.show()

In [ ]:
def __plot_gwb(fobs, gwb, hc_ss=None, bglabel=None, sslabel=None, **kwargs):
    xx = fobs * YR
    fig, ax = figax(
        xlabel=LABEL_GW_FREQUENCY_YR,
        ylabel=LABEL_CHARACTERISTIC_STRAIN
    )
    if(hc_ss is not None):
        draw_ss_and_gwb(ax, xx, hc_ss, gwb, sslabel=sslabel,
                        bglabel=bglabel, **kwargs)
    else:
        draw_gwb(ax, xx, gwb, **kwargs)
    _twin_hz(ax)
    return fig

def __draw_gwb(ax, xx, gwb, nsamp=10, color=None, label=None, alpha=0.25, **kwargs):
    if color is None:
        color = ax._get_lines.get_next_color()

    print(f"{len(gwb)=}")
    print(f"{gwb.shape=}")
    kw_plot = kwargs.pop('plot', {})
    kw_plot.setdefault('color', color)
    hh = __draw_med_conf(ax, xx, gwb, plot=kw_plot, label=label, **kwargs)
    if (nsamp is not None) and (nsamp > 0):
        nsamp_max = gwb.shape[1]
        idx = np.random.choice(nsamp_max, np.min([nsamp, nsamp_max]), replace=False)
        for ii in idx:
            ax.plot(xx, gwb[:, ii], color=color, alpha=alpha, lw=1.0, ls='-')

    return hh

def __draw_med_conf(ax, xx, vals, fracs=[0.50, 0.90], weights=None, plot={}, 
                    fill={}, filter=False, label=None, lw=1.0, ls='-'):
    #plot.setdefault('alpha', 0.75)
    #fill.setdefault('alpha', 0.2)
    plot.setdefault('alpha', 0.75)
    fill.setdefault('alpha', 0.1)
    percs = np.atleast_1d(fracs)
    assert np.all((0.0 <= percs) & (percs <= 1.0))

    # center the target percentages into pairs around 50%, e.g.  68 ==> [16,84]
    inter_percs = [[0.5-pp/2, 0.5+pp/2] for pp in percs]
    # Add the median value (50%)
    inter_percs = [0.5, ] + np.concatenate(inter_percs).tolist()
    # Get percentiles; they go along the last axis
    if filter:
        rv = [
            kale.utils.quantiles(vv[vv > 0.0], percs=inter_percs, weights=weights)
            for vv in vals
        ]
        rv = np.asarray(rv)
    else:
        rv = kale.utils.quantiles(vals, percs=inter_percs, weights=weights, axis=-1)

    med, *conf = rv.T
    # plot median
    hh, = ax.plot(xx, med, **plot, lw=lw, ls=ls, label=label)

    # Reshape confidence intervals to nice plotting shape
    # 2*P, X ==> (P, 2, X)
    conf = np.array(conf).reshape(len(percs), 2, xx.size)

    kw = dict(color=hh.get_color())
    kw.update(fill)
    fill = kw

    # plot each confidence interval
    for lo, hi in conf:
        gg = ax.fill_between(xx, lo, hi, **fill)

    return (hh, gg)

def compare_gwb_sim_vs_sam(sams, dpops, gpf_flags=None, 
                           save=True, fname_extra='',sam_colors=None,sam_lbls=None):

    LABEL_GW_FREQUENCY_YR = r"GW Frequency $[\mathrm{yr}^{-1}]$"
    LABEL_GW_FREQUENCY_HZ = r"GW Frequency $[\mathrm{Hz}]$"
    LABEL_GW_FREQUENCY_NHZ = r"GW Frequency $[\mathrm{nHz}]$"
    LABEL_SEPARATION_PC = r"Binary Separation $[\mathrm{pc}]$"
    LABEL_CHARACTERISTIC_STRAIN = r"GW Characteristic Strain"
    LABEL_HARDENING_TIME = r"Hardening Time $[\mathrm{Gyr}]$"
    LABEL_CLC0 = r"$C_\ell / C_0$"

    fig, ax = plot.figax(
        xlabel=LABEL_GW_FREQUENCY_YR,
        ylabel=LABEL_CHARACTERISTIC_STRAIN
    )

    frac = 0.50

    if len(sams) > 0:
        sam_freqs = sams[0].PARS['freqs']
        xx = sam_freqs * YR
        if sam_colors is None: 
            sam_colors = ['k']*len(sams)
        for k,s in enumerate(sams):
            lbl = s.model_type
            if s.model_type == 'old':
                lw=1.5
                col='k'
                ls='-'
            elif s.model_type == 'old_rc100':
                lw=2.5
                col='k'
                ls=':' if len(dpops)==0 else '-'
            elif s.model_type == 'ph15':
                lw=2.5 if len(dpops)==0 else 2
                col='darkred' if len(dpops)==0 else 'k'
                ls='-.'
            elif s.model_type == 'old_2s':
                lw=2
                col='k'
                ls='--'
            elif s.model_type == 'astr_rc100':
                lw=2.5 if len(dpops)==0 else 3
                col='r' if len(dpops)==0 else 'k'
                ls='-'
            elif s.model_type == 'astr_nuo0':
                lw=2
                col='darkgray'
                ls='-.'
            elif s.model_type == 'astr':
                lw=2
                col='darkgray'
                ls='-' if len(dpops)==0 else ':'
            else:
                raise ValueError(f"havent defined plot style for {s.model_type=}")

            gpf_lbls = [' (GMR)', ' (GPF)']
            if len(gpf_flags) == len(sams):
                lbl += gpf_lbls[gpf_flags[k]] 
                ls = '--' if gpf_flags[k] else '-'
            s_hctot = calc_hctot(s.gwb_sam)
            #__draw_gwb(ax, xx, s_hctot, nsamp=0, color=sam_colors[k], label=lbl,
            __draw_gwb(ax, xx, s_hctot, nsamp=0, color=col, label=lbl,
                       lw=lw, ls=ls, alpha=0.05, fracs=[frac])

    if len(dpops) > 0:
        for j,d in enumerate(dpops):
            xx = d.freqs * YR
            #print(f"{d.lbl=}, {d.gwb.both.shape=}")
            #col = 'b' if 'bh0' in d.lbl else d.color
            col = 'c' if 'bh0' in d.lbl else d.color
            lw = 1+0.5*j if 'bh0' in d.lbl else 1.5+0.25*j
            #ls = ':' if 'bh0' in d.lbl else '-.'
            #if d.hard_model_type == 'old_rc100':
            if d.hard_model_type == 'ph15':
                #ls = '-'
                ls = '--' if 'rTNG300' in d.lbl else '-'
                __draw_gwb(ax, xx, d.gwb.both, nsamp=0, color=col, 
                           label=d.lbl, lw=lw, ls=ls, alpha=0.05, fracs=[frac])
            else:
                #ls = ':'
                ls = '--' if 'rTNG300' in d.lbl else ':'
                __draw_gwb(ax, xx, d.gwb.both, nsamp=0, color=col, 
                           lw=lw, ls=ls, alpha=0.05, fracs=[frac])
                
    plt.legend(loc='lower left',fontsize=9)
    plt.suptitle(fname_extra)
    if save:
        #plt.savefig('gwb_compare_ill_tng100_main.png',dpi=300)
        if fname_extra != '':
            fname = f'gwb_compare_nloud{NLOUD}_nreals{NREALS}_tau{TAU}_{fname_extra}.png'
        else:
            fname = f'gwb_compare_nloud{NLOUD}_nreals{NREALS}_tau{TAU}.png'            
        plt.savefig(fname, dpi=300)


In [ ]:
def plot_bin_pop(pop, lbl, save=False):
    mt, mr = utils.mtmr_from_m1m2(pop.mass)
    print(f"{pop.mass.shape=}")
    m2 = pop.mass.min(axis=1)
    redz = cosmo.a_to_z(pop.scafa)
    #data = [mt/MSOL, mr, pop.sepa/PC, 1+redz]
    data = [mt/MSOL, mr, m2/MSOL, 1+redz]
    data = [np.log10(dd) for dd in data]
    reflect = [None, [None, 0], None, [0, None]]
    ###labels = [r'M/M_\odot', 'q', r'a/\mathrm{{pc}}', '1+z']
    labels = [r'Mtot/M_\odot', 'q', r'M2/M_\odot', '1+z']
    labels = [r'${{\log_{{10}}}} \left({}\right)$'.format(ll) for ll in labels]

    if pop.eccen is not None:
        data.append(pop.eccen)
        reflect.append([0.0, 1.0])
        labels.append('e')

    kde = kale.KDE(data, reflect=reflect)
    corner = kale.Corner(kde, labels=labels, figsize=[8, 8])
    corner.plot_data(kde)
    plt.suptitle(lbl)
    if save: plt.savefig(f'binary_properties_corner_{lbl}.png',dpi=300)
    return corner

def compare_bin_pops(dpops, labels=None, colors=None, lws=None, density=True, 
                     hist=False, confidence=False, fname_extra='', save=False):
    
    assert isinstance(dpops, list), '`dpops` must be a list of discrete populations'

    if (colors is not None):
        if (len(colors) != len(dpops)):
            print("Warning: `colors` must be a list of length len(dpops). Setting to None.")
            colors = None

    fig, axs = plt.subplots(nrows=3, ncols=2, figsize=[10,10])

    #axs[0].grid(alpha=0.01)
    ylbl = 'Probability Density'

    # indices are [row, column]
    axs[0,0].set(xlabel=r'$\log_{10}(M_{tot})$', ylabel=ylbl)
    axs[0,1].set(xlabel=r'$q$', ylabel=ylbl)
    
    axs[1,0].set(xlabel=r'$\log_{10}(M_{1})$', ylabel=ylbl)
    axs[1,1].set(xlabel=r'$\log_{10}(M_{2})$', ylabel=ylbl)

    axs[2,0].set(xlabel=r'$\log_{10}(1+z_{\rm init})$', ylabel=ylbl)

    for i,dp in enumerate(dpops):

        mt, mr = utils.mtmr_from_m1m2(dp.pop.mass)
        m1 = dp.pop.mass.max(axis=1)
        m2 = dp.pop.mass.min(axis=1)
        redz = cosmo.a_to_z(dp.pop.scafa)                
                
        #ls='--' if 'bh0' in dp.lbl else '-'
        ls = '--' if 'rTNG300' in d.lbl else '-'
        col = dp.color if colors is None else colors[i]
        
        kale.dist1d(np.log10(mt/MSOL), ax=axs[0,0], density=density, hist=hist, confidence=confidence, 
                    label=dp.lbl, color=col, lw=dp.lw, ls=ls, alpha=0.5)
        kale.dist1d(np.log10(mr), ax=axs[0,1], density=density, hist=hist, confidence=confidence, 
                    label=dp.lbl, color=col, lw=dp.lw, ls=ls, alpha=0.5)
        kale.dist1d(np.log10(m1/MSOL), ax=axs[1,0], density=density, hist=hist, confidence=confidence, 
                    label=dp.lbl, color=col, lw=dp.lw, ls=ls, alpha=0.5)
        kale.dist1d(np.log10(m2/MSOL), ax=axs[1,1], density=density, hist=hist, confidence=confidence, 
                    label=dp.lbl, color=col, lw=dp.lw, ls=ls, alpha=0.5)
        kale.dist1d(np.log10(1+redz), ax=axs[2,0], density=density, hist=hist, confidence=confidence, 
                    label=dp.lbl, color=col, lw=dp.lw, ls=ls, alpha=0.5)
        axs[0,1].legend()
        
    if save:
        fig.savefig(f'compare_binary_pars_{fname_extra}.png', dpi=300)


In [ ]:
def plot_loud_binary_params(sam, dpop, gpf_flag=0, save=False):
    
    #fig, axs = plt.subplots(nrows=3, ncols=3, sharex=True, figsize=[10,8])
    fig, axs = plt.subplots(nrows=2, ncols=3, sharex=True, figsize=[10,5])

    bg_alpha = 0.1
    ss_alpha = 0.3
    nskip = 1
    
    ##sam_hcss = sam.gwb_sam[0]    # shape: [nfreqs, nreals, nloudest]
    ##sam_hcbg = sam.gwb_sam[1]    # shape: [nfreqs, nreals]
    ##gwb_sam_gmr_hctot = np.sqrt( np.sum(gwb_sam_gmr[0]**2,axis=2) + gwb_sam_gmr[1]**2 )
    freqs_sam = s.PARS['freqs']
    sam_sspars = sam.gwb_sam[2]  # shape: [nsspars, nfreqs, nreals, nloudest]
    sam_bgpars = sam.gwb_sam[3]  # shape: [nfreqs, nreals]
    
    sam_bg_m1, sam_bg_m2 = utils.m1m2_from_mtmr(sam_bgpars[0,:,:], sam_bgpars[1,:,:])
    sam_ss_m1, sam_ss_m2 = utils.m1m2_from_mtmr(sam_sspars[0,:,:,:], sam_sspars[1,:,:,:])
    
    dpop_ss_m1, dpop_ss_m2 = utils.m1m2_from_mtmr(dpop.gwb.sspar[0], dpop.gwb.sspar[1])
    dpop_bg_m1, dpop_bg_m2 = utils.m1m2_from_mtmr(dpop.gwb.bgpar[0], dpop.gwb.bgpar[1])
    print(f"{dpop.gwb.bgpar[0].shape=}")
    
    # avg binary params of background sources for each freq. for each realization
    # 1st column panels
    axs[0,0].set(ylabel='Mtot',xscale='log',yscale='log')
    axs[0,0].plot(freqs_sam, sam_bgpars[0,:,::nskip]/MSOL,'gray',lw=0.2,alpha=bg_alpha); 
    axs[0,0].plot(dpop.freqs, dpop.gwb.bgpar[0][:, ::nskip]/MSOL,'c',lw=0.2,alpha=bg_alpha);

    axs[1,0].set(ylabel='q',xlabel='frequency', xscale='log') 
    axs[1,0].plot(freqs_sam, sam_bgpars[1,:,::nskip],'gray',lw=0.2,alpha=bg_alpha);
    axs[1,0].plot(dpop.freqs, dpop.gwb.bgpar[1][:, ::nskip],'c',lw=0.2,alpha=bg_alpha); 

    #axs[2,0].set(ylabel='Nbin',xlabel='frequency', xscale='log') 
    ##axs[2,0].plot(freqs_sam, sam_bgpars[7,:,::nskip],'gray',lw=0.2,alpha=bg_alpha);
    #axs[2,0].plot(dpop.freqs, dpop.gwb.bgpar[4][:, ::nskip],'c',lw=0.2,alpha=bg_alpha); 


    # 2nd column panels
    axs[0,1].set(ylabel='M1',xscale='log',yscale='log') #xlabel='frequency', 
    axs[0,1].plot(freqs_sam, sam_bg_m1[:, ::nskip]/MSOL,'gray',lw=0.2,alpha=bg_alpha); 
    axs[0,1].plot(dpop.freqs, dpop_bg_m1[:, ::nskip]/MSOL,'c',lw=0.2,alpha=bg_alpha); 

    axs[1,1].set(ylabel='M2',xlabel='frequency', xscale='log',yscale='log') 
    axs[1,1].plot(freqs_sam, sam_bg_m2[:, ::nskip]/MSOL,'gray',lw=0.2,alpha=bg_alpha); 
    axs[1,1].plot(dpop.freqs, dpop_bg_m2[:, ::nskip]/MSOL,'c',lw=0.2,alpha=bg_alpha);

    # 3rd column panels
    axs[0,2].set(ylabel='z_init',xscale='log', ylim=(0,2.5)) #,yscale='log')
    axs[0,2].plot(freqs_sam, sam_bgpars[2,:,::nskip],'gray',lw=0.2,alpha=bg_alpha); 
    axs[0,2].plot(dpop.freqs, dpop.gwb.bgpar[2][:, ::nskip],'c',lw=0.2,alpha=bg_alpha); 

    axs[1,2].set(xlabel='frequency', ylabel='z_final',xscale='log', ylim=(0,2.5))
    axs[1,2].plot(freqs_sam, sam_bgpars[3,:,::nskip],'gray',lw=0.2,alpha=bg_alpha); 
    axs[1,2].plot(dpop.freqs, dpop.gwb.bgpar[3][:, ::nskip],'c',lw=0.2,alpha=bg_alpha); 

    print(dpop.gwb.sspar[3].shape) ##nfreq, nloud, nreals

    for ll in range(NLOUD):
        # 1st column panels
        # mt of 10 loudest sources in each freq bin, avg over nreals
        axs[0,0].plot(freqs_sam, np.mean(sam_sspars[0,:,:,ll],axis=1)/MSOL, 'k.', alpha=ss_alpha)
        axs[0,0].plot(dpop.freqs, np.mean(dpop.gwb.sspar[0][:,ll,:],axis=1)/MSOL, 'b.', alpha=ss_alpha) 

        # mrat of 10 loudest sources in each freq bin, avg over nreals
        axs[1,0].plot(freqs_sam, np.mean(sam_sspars[1,:,:,ll],axis=1), 'k.', alpha=ss_alpha) 
        axs[1,0].plot(dpop.freqs, np.mean(dpop.gwb.sspar[1][:,ll,:],axis=1), 'b.', alpha=ss_alpha)

        # 2nd column panels
        # m1 of 10 loudest sources in each freq bin, avg over nreals
        axs[0,1].plot(freqs_sam, np.mean(sam_ss_m1[:,:,ll],axis=1)/MSOL, 'k.', alpha=ss_alpha)
        axs[0,1].plot(dpop.freqs, np.mean(dpop_ss_m1[:,ll,:],axis=1)/MSOL, 'b.', alpha=ss_alpha) 

        # m2 of 10 loudest sources in each freq bin, avg over nreals
        axs[1,1].plot(freqs_sam, np.mean(sam_ss_m2[:,:,ll],axis=1)/MSOL, 'k.', alpha=ss_alpha) 
        axs[1,1].plot(dpop.freqs, np.mean(dpop_ss_m2[:,ll,:],axis=1)/MSOL, 'b.', alpha=ss_alpha)

        # 3rd column panels
        # rzi of 10 loudest sources in each freq bin, avg over nreals
        axs[0,2].plot(freqs_sam, np.mean(sam_sspars[2,:,:,ll],axis=1), 'k.', alpha=ss_alpha) 
        axs[0,2].plot(dpop.freqs, np.mean(dpop.gwb.sspar[2][:,ll,:],axis=1), 'b.', alpha=ss_alpha) 

        # rzf of 10 loudest sources in each freq bin, avg over nreals
        sam_lbl = 'SAM' if ll==0 else None
        sim_lbl = 'sim' if ll==0 else None
        axs[1,2].plot(freqs_sam, np.mean(sam_sspars[3,:,:,ll],axis=1), 'k.', alpha=ss_alpha, label=sam_lbl)
        axs[1,2].plot(dpop.freqs, np.mean(dpop.gwb.sspar[3][:,ll,:],axis=1), 'b.', alpha=ss_alpha, label=sim_lbl) 

        
    axs[1,2].legend(loc='upper left')
    
    fname_extra = 'gpf' if gpf_flag else 'gmr'

    plt_title = f'{dpop.lbl} ({dpop.hard_model_type}) vs {fname_extra} SAM ({sam.model_type})'
    fig.suptitle(plt_title)
    plt.subplots_adjust(hspace=0.1,wspace=0.3,top=0.92)

    if save: 
        fig.savefig(f'loud_bin_params_nloud{NLOUD}_nreals{NREALS}_tau{TAU}_{dpop.lbl}_{fname_extra}.png', dpi=300)


In [ ]:
def plot_evo(evo, freqs=None, sepa=None, ax=None, label=None, color=None, **kwargs):
    if (freqs is None) and (sepa is None):
        err = "Either `freqs` or `sepa` must be provided!"
        log.exception(err)
        raise ValueError(err)

    if freqs is not None:
        data = evo.at('fobs', freqs)
        xx = freqs * YR
        xlabel = 'GW Frequency [1/yr]'
    else:
        data = evo.at('sepa', sepa)
        xx = sepa / PC
        xlabel = 'Binary Separation [pc]'

    if ax is None:
        fig, ax = plot.figax(xlabel=xlabel)
    else:
        fig = ax.get_figure()

    def _draw_vals_conf(ax, xx, vals, color=color, label=label):
        if color is None:
            color = ax._get_lines.get_next_color()
        if label is not None:
            ax.set_ylabel(label, color=color)
            ax.tick_params(axis='y', which='both', colors=color)
        # vals = np.percentile(vals, [25, 50, 75], axis=0) / units
        vals = utils.quantiles(vals, [0.25, 0.50, 0.75], axis=0).T
        h1 = ax.fill_between(xx, vals[0], vals[-1], alpha=0.2, color=color)
        h2, = ax.plot(xx, vals[1], alpha=0.5, lw=2.0, color=color)
        return (h1, h2)

    # handles = []
    # labels = []

    name = 'Hardening Time [yr]'
    vals = np.fabs(data['sepa'] / data['dadt']) / YR
    _draw_vals_conf(ax, xx, vals, label=name)
    # handles.append(hh)
    # labels.append(name)

    # name = 'eccen'
    # tw = ax.twinx()
    # hh, nn = _draw_vals_conf(tw, freqs*YR, name, 'green')
    # if hh is not None:
    #     handles.append(hh)
    #     labels.append(nn)

    # ax.legend(handles, labels)
    return ax

# doesn't work, need to write a proper function to plot sam hardening
#def plot_sam_hardening_tscale(testsam):
    
#    dadt = testsam.hard.dadt(testsam.sam.mtot, testsam.sam.mrat, 
#                             testsam.sam.sepa, norm=None)
#    print(dadt.shape)


# Load the sim and SAM data

In [ ]:
def load_sim_data(nloud=1, nreals=10, nfreqs=40, tau_hard=1.0, sim='Illustris', 
                  hardmodel_name='old_rc100', mmodflag=True, t300rescale=False,
                  data_dir=_SIM_MERGER_PATH):
    
    if sim not in ['Illustris','TNG50','TNG100', 'TNG300']:
        raise ValueError(f"`sim` must be in ['Illustris','TNG50','TNG100','TNG300']")
    
    if t300rescale:
        sim += 'Rescale'
    modMMb = '_withModMMb' if mmodflag else ''
    resc = 'withT300Rescale' if t300rescale else ''
    
    # ---- Unpickle discrete pop data
    sim_pkl_fname = (f'dpops_nfreqs{nfreqs}_nreals{nreals}_nloud{nloud}_{sim}_'
                     f'hard{hardmodel_name}_tau{tau_hard}_fsaOnly{modMMb}{resc}.pkl')
    #if sim == 'TNG300':
    #    #sim_pkl_fname = f'dpops_nfreqs{nfreqs}_nreals{nreals}_nloud{nloud}_tau{tau_hard}_{sim}_withbh_fsaOnly{modMMb}.pkl'
    #    sim_pkl_fname = (f'dpops_nfreqs{nfreqs}_nreals{nreals}_nloud{nloud}_{sim}_'
    #                     f'hard{hardmodel_name}_tau{tau_hard}_fsaOnly{modMMb}.pkl')
    #else:
    #    #sim_pkl_fname = f'dpops_nfreqs{nfreqs}_nreals{nreals}_nloud{nloud}_compare_{sim}_cuts_fsaOnly{modMMb}.pkl'
    #    sim_pkl_fname = (f'dpops_nfreqs{nfreqs}_nreals{nreals}_nloud{nloud}_compare_{sim}_cuts_'
    #                     f'hard{hardmodel_name}_tau{tau_hard}_fsaOnly{modMMb}.pkl')
    #pkl_fname=f'dpops_nfreqs{NFREQS}_nreals{NREALS}_nloud{NLOUD}_tau{TAU}_compare_Illustris_cuts_with_fsa.pkl'
    #pkl_fname=f'dpops_nfreqs{nfreqs}_nreals{nreals}_nloud{nloud}_tau{tau_hard}_compare_Illustris_cuts_with_fsa.pkl'

    with open('/'.join((data_dir, sim_pkl_fname)), "rb") as f:
        print(f'unpickling dpop data: {sim_pkl_fname}')
        dp_data = pickle.load(f)
        for l in dp_data:
            if len(l) > 0:
                for i in range(len(l)):
                    #print(l[i])
                    print(l[i].lbl)
        
        # this clunky set of return values is from older code, should clean up at some point. 
        # we only need all_fsa_dpops.
        all_dpops, tng_dpops, all_fsa_dpops, tng_fsa_dpops = dp_data

    # THIS IS SKETCHY, FIX IT
    if not t300rescale:
        return all_fsa_dpops
    else:
        return tng_fsa_dpops
    
#def load_sam_data(nloud=1, nreals=10, nfreqs=40, tau_hard=1.0, gpf_flag=False):
def load_sam_data(nloud=1, nreals=10, nfreqs=40, gpf_flag=False, tau=1.0,
                  data_dir=_SIM_MERGER_PATH, fname_type='manual_moddefs'):

    if fname_type=='manual_moddefs':
        samtype = 'gpf' if gpf_flag else 'gmr'
    
        # ---- Unpickle SAM data
        ## right now these files are just stored in the same directory as the notebooks
        ## should move to data dir once things have stabilized a bit
        #sam_pkl_fname=f'sam_nfreqs{NFREQS}_nreals{NREALS}_nloud{NLOUD}_tau{TAU}_{samtype}.pkl'
        #sam_pkl_fname=f'sam_nfreqs{nfreqs}_nreals{nreals}_nloud{nloud}_model_type{model_type}_{samtype}.pkl'
        sam_pkl_fname=f'test_sam_nfreqs{nfreqs}_nreals{nreals}_nloud{nloud}_manual_moddefs_{samtype}_tau{tau}.pkl'

    elif fname_type=='old_new_mods_compare':
        sam_pkl_fname=f'test_sam_nfreqs{nfreqs}_nreals{nreals}_nloud{nloud}_old_new_mods_compare_tau5.55.pkl'
    else:
        raise ValueError(f'{fname_type=} not defined.')
    
    
    with open('/'.join((data_dir, sam_pkl_fname)), "rb") as f:
        print(f'unpickling SAM data: {sam_pkl_fname}')
        sams = pickle.load(f)
        #sam, hard, gwb_new_sam, gwb_sam, freqs, freqs_edges = sam_data
        #sam, hard, gwb_sam, MODEL_PARS = sam_data
   
    #return sam, hard, gwb_new_sam, gwb_sam, freqs, freqs_edges
    #return sam, hard, gwb_sam, MODEL_PARS
    return sams
    
    #gpf_pkl_fname=f'sam_nfreqs{NFREQS}_nreals{NREALS}_nloud{NLOUD}_tau{TAU}_gpf.pkl'
    #with open(gpf_pkl_fname, "rb") as f:
    #    print("unpickling GPF SAM data")
    #    gpf_data = pickle.load(f)
    #    sam_gpf, hard_gpf, gwb_new_sam_gpf, gwb_sam_gpf, freqs_gpf, freqs_edges_gpf = gpf_data

    #gmr_pkl_fname=f'sam_nfreqs{NFREQS}_nreals{NREALS}_nloud{NLOUD}_tau{TAU}_gmr.pkl'
    #with open(gmr_pkl_fname, "rb") as f:
    #    print("unpickling GMR SAM data")
    #    gmr_data = pickle.load(f)
    #    sam_gmr, hard_gmr, gwb_new_sam_gmr, gwb_sam_gmr, freqs_gmr, freqs_edges_gmr = gmr_data

def calc_hctot(gwb_sam):
    return np.sqrt( np.sum(gwb_sam[0]**2,axis=2) + gwb_sam[1]**2 )


In [ ]:
# ---- Load discrete pops (with mmod)
sim_hardmod_list = ['ph15']
#sim_hardmod_list = ['old_rc100', 'old_rc10', 'ph15', 'ph15_rc10']
tau_list = [1.0]
#tau_list = [1.0, 3.0]

ill_dpops_mmod = []
for hardmod in sim_hardmod_list:
    for t in tau_list:
    #for t in [1.0,3.0]:
        tmp = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=t, 
                            sim='Illustris', hardmodel_name=hardmod, mmodflag=True)
        ill_dpops_mmod += tmp

t50_dpops_mmod = []
for hardmod in sim_hardmod_list:
    for t in tau_list:
        tmp = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=t, 
                            sim='TNG50', hardmodel_name=hardmod, mmodflag=True)
        t50_dpops_mmod += tmp

t100_dpops_mmod = []
for hardmod in sim_hardmod_list:
    for t in tau_list:
        tmp = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=t, 
                            sim='TNG100', hardmodel_name=hardmod, mmodflag=True)
        t100_dpops_mmod += tmp
        
t300_dpops_mmod = []
for hardmod in sim_hardmod_list:
    for t in tau_list:
        tmp = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=t, 
                            sim='TNG300', hardmodel_name=hardmod, mmodflag=True)
        t300_dpops_mmod += tmp

        
print("LOADING RESCALE FILE")
t300_rescale_dpops_mmod = []
for hardmod in sim_hardmod_list:
    for t in tau_list:
        tmp = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=t, 
                            sim='TNG300', hardmodel_name=hardmod, mmodflag=True, t300rescale=True)
        t300_rescale_dpops_mmod += tmp
for d in t300_rescale_dpops_mmod:
    print("TEST:",d.lbl)

        
# ---- Load discrete pops (without mmod)
ill_dpops_nommod = []
for hardmod in sim_hardmod_list:
    for t in tau_list:
        tmp = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=t, 
                            sim='Illustris', hardmodel_name=hardmod, mmodflag=False)
        ill_dpops_nommod += tmp

t50_dpops_nommod = []
for hardmod in sim_hardmod_list:
    for t in tau_list:
        tmp = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=t, 
                            sim='TNG50', hardmodel_name=hardmod, mmodflag=False)
        t50_dpops_nommod += tmp

t100_dpops_nommod = []
for hardmod in sim_hardmod_list:
    for t in tau_list:
        tmp = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=t, 
                            sim='TNG100', hardmodel_name=hardmod, mmodflag=False)
        t100_dpops_nommod += tmp
        
t300_dpops_nommod = []
for hardmod in sim_hardmod_list:
    for t in tau_list:
        tmp = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=t, 
                            sim='TNG300', hardmodel_name=hardmod, mmodflag=False)
        t300_dpops_nommod += tmp

#t300_rescale_dpops_nommod = []
#for hardmod in sim_hardmod_list:
#    for t in tau_list:
#        tmp = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=t, 
#                            sim='TNG300', hardmodel_name=hardmod, mmodflag=False, t300rescale=True)
#        t300_rescale_dpops_nommod += tmp


#oldrc100_tau1 = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=1.0, 
#                                             sim='Illustris', hardmodel_name='old_rc100', mmodflag=True)
#ill_dpops_mmod_oldrc10_tau1 = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=1.0, 
#                                            sim='Illustris', hardmodel_name='old_rc10', mmodflag=True)
#ill_dpops_mmod_ph15_tau1 = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=1.0, 
#                                         sim='Illustris', hardmodel_name='ph15', mmodflag=True)
#ill_dpops_mmod_ph15_rc10_tau1 = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=1.0, 
#                                              sim='Illustris', hardmodel_name='ph15_rc10', mmodflag=True)
#tng100_dpops_mmod = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=TAU, 
#                                  sim='TNG100', mmodflag=True)
#tng50_dpops_mmod_oldrc100_tau1 = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=1.0, 
#                                               sim='TNG50', hardmodel_name='old_rc100', mmodflag=True)
#tng300_dpops_mmod = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=TAU, 
#                                  sim='TNG300', mmodflag=True)
#ill_dpops = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=TAU, 
#                          sim='Illustris', mmodflag=False)
#tng100_dpops = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=TAU, 
#                             sim='TNG100', mmodflag=False)
#tng50_dpops = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=TAU, 
#                            sim='TNG50', mmodflag=False)
#tng300_dpops = load_sim_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, tau_hard=TAU, 
#                            sim='TNG300', mmodflag=False)


In [ ]:

# ---- Load SAMs
sams_gmr_tau1 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
                              gpf_flag=False, tau=1.0, data_dir=_PATH_DATA)
#sams_gmr_tau3 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=False, tau=3.0, data_dir=_PATH_DATA)
#sams_gpf_tau1 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=True, tau=1.0, data_dir=_PATH_DATA)
#sams_gpf_tau3 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=True, tau=3.0, data_dir=_PATH_DATA)

for s in sams_gmr_tau1: #+sams_gmr_tau3+sams_gpf_tau1+sams_gpf_tau3:
    print(s.model_type)


In [ ]:
for d in ill_dpops_mmod_ph15_tau1:
    print(d.hard_model_type, d.HARD_PARS['hard_time'])
    if d.HARD_PARS['hard_time']==1: print('y')

In [ ]:
#gwb_amps(sams=sams_gmr_tau1, dpops=tng50_dpops_mmod_tau1, fname='test')
#gwb_amps(sams=None, dpops=ill_dpops_mmod_oldrc100_tau1+ill_dpops_mmod_oldrc10_tau1, fname='test')
#gwb_amps(sams=None, dpops=ill_dpops_mmod_oldrc100_tau1+ill_dpops_mmod_ph15, fname='test')
#gwb_amps(sams=None, dpops=ill_dpops_mmod_oldrc100_tau1+ill_dpops_mmod_ph15_rc10_tau1, fname='test')
gwb_amps(sams=None, dpops=ill_dpops_mmod_ph15_tau1+ill_dpops_mmod_ph15_rc10_tau1, fname='test')

In [ ]:
gwb_amps(sams=None, dpops=ill_dpops_mmod_oldrc10_tau1, fname='test')

In [ ]:
sepa = np.logspace(-4.5, 5, 100) * PC
fig, ax = plot.figax(figsize=(8,5))
for d in tng50_dpops_mmod_tau1:
    print(d.lbl)
    plot_evo(d.evo, freqs=None, sepa=sepa, ax=ax, label=d.lbl)
    #plt.title(d.lbl)

plt.legend()
plt.show()

# First sim vs SAM comparison: 
## How it started:
The GWB only cares about massive galaxies right? And those should all have BHs in them anyway, right?

Also, TNG100 has bigger volume than TNG50, so we should get more massive binaries that increase the GWB amplitude in TNG100, right?

Let's try comparing the GWB for sim mergers with min $N_{\rm DM}$, $N_{\rm star}$, & $N_{\rm gas}$ = 100, min $N_{\rm BH}$ = 1
...

## How it's going:

In [ ]:
# --- compare GWB amps for TNG50, TNG100, and Illustris using strict Npart cuts and native BH pop
lbl_list = ['fsa-Illustris-1','fsa-TNG50-1', 'fsa-TNG100-1', 'fsa-TNG300-1']
#lbl_list = ['fsa-Illustris-1'] #,'fsa-TNG50-1', 'fsa-TNG100-1', 'fsa-TNG300-1']
dp_list = [d for d in ill_dpops_nommod+t50_dpops_nommod+t100_dpops_nommod+t300_dpops_nommod 
           if d.lbl in lbl_list]
#dp_list = []
#dp_list = tng50_dpops_mmod_tau1

lbl_mmod_list = ['fsa-mm-Illustris-1','fsa-mm-TNG50-1', 'fsa-mm-TNG100-1', 'fsa-mm-TNG300-1']
dp_mmod_list = [d for d in ill_dpops_mmod+t50_dpops_mmod+t100_dpops_mmod+t300_dpops_mmod 
                if d.lbl in lbl_mmod_list]


#sam_mod_list = ['ph15','astr_rc100']
#sam_mod_list =['old','old_2s','old_rc100','ph15','astr', 'astr_nuo0','astr_rc100']
#sam_mod_list = ['ph15','astr_rc100','astr'] # high, med, low amps
#sam_tau_list = [1.0, 3.0]
sam_tau_list = [1.0]
#sam_tau_list = [3.0]
gmr_sam_mod_list = ['astr_rc100']
gpf_sam_mod_list = ['ph15']
gmr_sams_list = [ s for s in sams_gmr_tau1+sams_gmr_tau3
                  if (s.model_type in gmr_sam_mod_list and 
                  s.PARS['hard_time'] in sam_tau_list) ]
gpf_sams_list = [ s for s in sams_gpf_tau1+sams_gpf_tau3
                  if (s.model_type in gpf_sam_mod_list and 
                  s.PARS['hard_time'] in sam_tau_list) ]

sams_list = gmr_sams_list + gpf_sams_list
gpf_flags = [0]*len(gmr_sams_list) + [1]*len(gpf_sams_list) 
##gwb_amps(freqs_gpf, freqs_gmr, gwb_sam_gpf_hctot, gwb_sam_gmr_hctot, 
##         fname='compare_gwb_amps_nommod', dpops=dp_list)

gwb_amps(sams=sams_list, dpops=dp_list, fname='test_compare_gwb_amps',gpf_flags=gpf_flags)

#compare_gwb_sim_vs_sam(sams_gmr_tau1, dp_list, gpf_flags=[0]*len(sams_gmr_tau1), 
#                       fname_extra='test_gwb_compare_sams', save=False)
compare_gwb_sim_vs_sam(sams_list, dp_list, gpf_flags=gpf_flags, 
                       fname_extra='test_gwb_compare', save=False)

compare_bin_pops(dp_list, fname_extra='test', colors=['g','r','b','m'])

print(gmr_sams_list[0].model_type, dp_list[0].hard_model_type)
plot_loud_binary_params(gmr_sams_list[0], dp_list[0], gpf_flag=0, save=False)

gwb_amps(sams=sams_list, dpops=dp_mmod_list, fname='test_compare_gwb_amps',gpf_flags=gpf_flags)

#compare_gwb_sim_vs_sam(sams_gmr_tau1, dp_list, gpf_flags=[0]*len(sams_gmr_tau1), 
#                       fname_extra='test_gwb_compare_sams', save=False)
compare_gwb_sim_vs_sam(sams_list, dp_mmod_list, gpf_flags=gpf_flags, 
                       fname_extra='test_gwb_compare', save=False)

compare_bin_pops(dp_mmod_list, fname_extra='test', colors=['g','r','b','m'])

print(gmr_sams_list[0].model_type, dp_mmod_list[0].hard_model_type)
plot_loud_binary_params(gmr_sams_list[0], dp_mmod_list[0], gpf_flag=0, save=False)

print(gpf_sams_list[0].model_type, dp_mmod_list[0].hard_model_type)
plot_loud_binary_params(gpf_sams_list[0], dp_mmod_list[0], gpf_flag=1, save=False)

print(gmr_sams_list[0].model_type, dp_mmod_list[1].hard_model_type)
plot_loud_binary_params(gmr_sams_list[0], dp_mmod_list[1], gpf_flag=0, save=False)

print(gmr_sams_list[0].model_type, dp_mmod_list[2].hard_model_type)
plot_loud_binary_params(gmr_sams_list[0], dp_mmod_list[2], gpf_flag=0, save=False)

print(gmr_sams_list[0].model_type, dp_mmod_list[3].hard_model_type)
plot_loud_binary_params(gmr_sams_list[0], dp_mmod_list[3], gpf_flag=0, save=False)


In [ ]:

lbl_mmod_list = ['fsa-mm-Illustris-1','fsa-mm-TNG50-1', 'fsa-mm-TNG100-1', 'fsa-mm-TNG300-1']
dp_mmod_list = [d for d in ill_dpops_mmod+t50_dpops_mmod+t100_dpops_mmod+t300_dpops_mmod
                if d.lbl in lbl_mmod_list]
dp_mmod_list += [d for d in t300_rescale_dpops_mmod if d.lbl=='fsa-mm-rTNG300-1']
#for d in dp_mmod_list:
for d in t300_rescale_dpops_mmod:
    print(d.lbl)
#t300_rescale_dpops_mmod 
#sam_mod_list = ['ph15','astr_rc100']
#sam_mod_list =['old','old_2s','old_rc100','ph15','astr', 'astr_nuo0','astr_rc100']
#sam_mod_list = ['ph15','astr_rc100','astr'] # high, med, low amps
#sam_tau_list = [1.0, 3.0]
sam_tau_list = [1.0]
#sam_tau_list = [3.0]
gmr_sam_mod_list = ['astr_rc100']
gpf_sam_mod_list = ['ph15']
gmr_sams_list = [ s for s in sams_gmr_tau1+sams_gmr_tau3
                  if (s.model_type in gmr_sam_mod_list and 
                  s.PARS['hard_time'] in sam_tau_list) ]
gpf_sams_list = [ s for s in sams_gpf_tau1+sams_gpf_tau3
                  if (s.model_type in gpf_sam_mod_list and 
                  s.PARS['hard_time'] in sam_tau_list) ]

sams_list = gmr_sams_list + gpf_sams_list
gpf_flags = [0]*len(gmr_sams_list) + [1]*len(gpf_sams_list) 
##gwb_amps(freqs_gpf, freqs_gmr, gwb_sam_gpf_hctot, gwb_sam_gmr_hctot, 
##         fname='compare_gwb_amps_nommod', dpops=dp_list)


gwb_amps(sams=sams_list, dpops=dp_mmod_list, fname='test_compare_gwb_amps',gpf_flags=gpf_flags)

#compare_gwb_sim_vs_sam(sams_gmr_tau1, dp_list, gpf_flags=[0]*len(sams_gmr_tau1), 
#                       fname_extra='test_gwb_compare_sams', save=False)
compare_gwb_sim_vs_sam(sams_list, dp_mmod_list, gpf_flags=gpf_flags, 
                       fname_extra='test_gwb_compare', save=False)

compare_bin_pops(dp_mmod_list, fname_extra='test', colors=['g','r','b','m','m'])



In [ ]:
# --- compare GWB amps for different sim resolutions

lbl_list = ['fsa-Illustris-1','fsa-TNG300-1','fsa-TNG100-1','fsa-TNG100-2',
                 'fsa-TNG50-1','fsa-TNG50-2','fsa-TNG50-3']
dp_list = [d for d in ill_dpops_nommod+t50_dpops_nommod+t100_dpops_nommod+t300_dpops_nommod 
                if d.lbl in lbl_list]


lbl_mmod_list = ['fsa-mm-Illustris-1','fsa-mm-TNG300-1','fsa-mm-TNG100-1','fsa-mm-TNG100-2',
                 'fsa-mm-TNG50-1','fsa-mm-TNG50-2','fsa-mm-TNG50-3']
dp_mmod_list = [d for d in ill_dpops_mmod+t50_dpops_mmod+t100_dpops_mmod+t300_dpops_mmod 
                if d.lbl in lbl_mmod_list]


#sam_mod_list = ['ph15','astr_rc100']
#sam_mod_list =['old','old_2s','old_rc100','ph15','astr', 'astr_nuo0','astr_rc100']
#sam_mod_list = ['ph15','astr_rc100','astr'] # high, med, low amps
#sam_tau_list = [1.0, 3.0]
sam_tau_list = [1.0]
#sam_tau_list = [3.0]
gmr_sam_mod_list = ['astr_rc100']
gpf_sam_mod_list = ['ph15']
gmr_sams_list = [ s for s in sams_gmr_tau1+sams_gmr_tau3
                  if (s.model_type in gmr_sam_mod_list and 
                  s.PARS['hard_time'] in sam_tau_list) ]
gpf_sams_list = [ s for s in sams_gpf_tau1+sams_gpf_tau3
                  if (s.model_type in gpf_sam_mod_list and 
                  s.PARS['hard_time'] in sam_tau_list) ]

sams_list = gmr_sams_list + gpf_sams_list
gpf_flags = [0]*len(gmr_sams_list) + [1]*len(gpf_sams_list) 
##gwb_amps(freqs_gpf, freqs_gmr, gwb_sam_gpf_hctot, gwb_sam_gmr_hctot, 
##         fname='compare_gwb_amps_nommod', dpops=dp_list)

gwb_amps(sams=sams_list, dpops=dp_list, fname='test_compare_gwb_amps',gpf_flags=gpf_flags)

#compare_gwb_sim_vs_sam(sams_gmr_tau1, dp_list, gpf_flags=[0]*len(sams_gmr_tau1), 
#                       fname_extra='test_gwb_compare_sams', save=False)
compare_gwb_sim_vs_sam(sams_list, dp_list, gpf_flags=gpf_flags, 
                       fname_extra='test_gwb_compare', save=False)

compare_bin_pops(dp_list, fname_extra='test', colors=['g','r','b','m'])

#for dp in dp_list:
#    plot_loud_binary_params(freqs_gmr, gwb_sam_gmr_bgpars, gwb_sam_gmr_sspars, 
#                            dp, gpf_flag=0, save=True)

gwb_amps(sams=sams_list, dpops=dp_mmod_list, fname='test_compare_gwb_amps',gpf_flags=gpf_flags)

#compare_gwb_sim_vs_sam(sams_gmr_tau1, dp_list, gpf_flags=[0]*len(sams_gmr_tau1), 
#                       fname_extra='test_gwb_compare_sams', save=False)
compare_gwb_sim_vs_sam(sams_list, dp_mmod_list, gpf_flags=gpf_flags, 
                       fname_extra='test_gwb_compare', save=False)

compare_bin_pops(dp_mmod_list, fname_extra='test', colors=['g','r','b','m'])



In [ ]:
#print([0,1]*4)
foo = [1,2,3,4]
bar = [n for n in foo for _ in range(2)]
print(bar)

In [ ]:
sam_mod_list =['old','old_2s','old_rc100','ph15','astr', 'astr_nuo0','astr_rc100']
sam_mod_list = ['ph15','astr_rc100','astr'] # high, med, low amps
#sam_tau_list = [1.0, 3.0]
sam_tau_list = [1.0]
#sam_tau_list = [3.0]
gmr_sams_list = [ s for s in sams_gmr_tau1 #+sams_gmr_tau3
                  if (s.model_type in sam_mod_list and 
                  s.PARS['hard_time'] in sam_tau_list) ]
#gpf_sams_list = [ s for s in sams_gpf_tau1+sams_gpf_tau3
#                  if (s.model_type in sam_mod_list and 
#                  s.PARS['hard_time'] in sam_tau_list) ]

for s in gmr_sams_list:
    print(s.model_type)
sams_list = gmr_sams_list 
gpf_flags = [0]*len(gmr_sams_list) 
#sams_list = gmr_sams_list + gpf_sams_list
#gpf_flags = [0]*len(gmr_sams_list) + [1]*len(gpf_sams_list) 

#gwb_amps(sams=sams_list, dpops=None, fname='compare_amps_gmr_gpf_tau1', color_hardmods=False)
#compare_gwb_sim_vs_sam(sams_list, [], gpf_flags=gpf_flags, 
#                       fname_extra='compare_gwb_gmr_gpf_tau1', save=False)
#compare_gwb_sim_vs_sam(sams_list, [], gpf_flags=gpf_flags, 
#                       fname_extra='compare_gwb_gmr_gpf_tau3', save=False)

# ---- GMR TAU=1
gwb_amps(sams=sams_list, dpops=None, fname='compare_amps_gmr_tau1', 
         color_hardmods=False, gpf_flags=gpf_flags)

compare_gwb_sim_vs_sam(sams_list, [], gpf_flags=gpf_flags, 
                       fname_extra='compare_gwb_gmr_tau1', save=False)
## ---- GMR TAU=3
#gwb_amps(sams=sams_gmr_tau3, dpops=None, fname='compare_amps_gmr_tau3', color_hardmods=False)

#compare_gwb_sim_vs_sam(sams_gmr_tau3, [], gpf_flags=[0]*len(sams_gmr_tau1), 
#                       fname_extra='compare_gwb_gmr_tau3', save=False)
## ---- GPF TAU=1
#gwb_amps(sams=sams_gpf_tau1, dpops=None, fname='compare_amps_gpf_tau1', color_hardmods=False)

#compare_gwb_sim_vs_sam(sams_gpf_tau1, [], gpf_flags=[1]*len(sams_gpf_tau1), 
#                       fname_extra='compare_gwb_gpf_tau1', save=False)
## ---- GPF TAU=3
#gwb_amps(sams=sams_gpf_tau3, dpops=None, fname='compare_amps_gpf_tau3', color_hardmods=False)

#compare_gwb_sim_vs_sam(sams_gpf_tau3, [], gpf_flags=[1]*len(sams_gpf_tau1), 
#                       fname_extra='compare_gwb_gpf_tau3', save=False)


In [ ]:

# ---- Load SAMs
sams_old_new_mods_compare = load_sam_data(nloud=NLOUD, nreals=50, nfreqs=NFREQS, 
                                          gpf_flag=None, tau=None, data_dir=_PATH_DATA,
                                          fname_type='old_new_mods_compare')
#sams_gmr_tau3 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=False, tau=3.0, data_dir=_PATH_DATA)
#sams_gpf_tau1 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=True, tau=1.0, data_dir=_PATH_DATA)
#sams_gpf_tau3 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=True, tau=3.0, data_dir=_PATH_DATA)

for s in sams_old_new_mods_compare: #+sams_gmr_tau3+sams_gpf_tau1+sams_gpf_tau3:
    print(s.model_type)

gpf_flags = [1, 0]

gwb_amps(sams=sams_old_new_mods_compare, dpops=None, fname='compare_amps_sams_old_new_mods_compare', 
         color_hardmods=False, gpf_flags=gpf_flags)

compare_gwb_sim_vs_sam(sams_old_new_mods_compare, [], gpf_flags=gpf_flags, 
                       fname_extra='compare_gwb__sams_old_new_mods_compare', save=False)


In [ ]:

# ---- Load SAMs
sams_old_new_mods_compare = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
                                          gpf_flag=None, tau=None, data_dir=_PATH_DATA)
#sams_gmr_tau3 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=False, tau=3.0, data_dir=_PATH_DATA)
#sams_gpf_tau1 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=True, tau=1.0, data_dir=_PATH_DATA)
#sams_gpf_tau3 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=True, tau=3.0, data_dir=_PATH_DATA)

for s in _old_new_mods_compare: #+sams_gmr_tau3+sams_gpf_tau1+sams_gpf_tau3:
    print(s.model_type)

sam_mod_list =['old','old_2s','old_rc100','ph15','astr', 'astr_nuo0','astr_rc100']
sam_mod_list = ['ph15','astr_rc100','astr'] # high, med, low amps
#sam_tau_list = [1.0, 3.0]
sam_tau_list = [1.0]
#sam_tau_list = [3.0]
gmr_sams_list = [ s for s in sams_gmr_tau1 #+sams_gmr_tau3
                  if (s.model_type in sam_mod_list and 
                  s.PARS['hard_time'] in sam_tau_list) ]
#gpf_sams_list = [ s for s in sams_gpf_tau1+sams_gpf_tau3
#                  if (s.model_type in sam_mod_list and 
#                  s.PARS['hard_time'] in sam_tau_list) ]

for s in gmr_sams_list:
    print(s.model_type)
sams_list = gmr_sams_list 
gpf_flags = [0]*len(gmr_sams_list) 
#sams_list = gmr_sams_list + gpf_sams_list
#gpf_flags = [0]*len(gmr_sams_list) + [1]*len(gpf_sams_list) 

#gwb_amps(sams=sams_list, dpops=None, fname='compare_amps_gmr_gpf_tau1', color_hardmods=False)
#compare_gwb_sim_vs_sam(sams_list, [], gpf_flags=gpf_flags, 
#                       fname_extra='compare_gwb_gmr_gpf_tau1', save=False)
#compare_gwb_sim_vs_sam(sams_list, [], gpf_flags=gpf_flags, 
#                       fname_extra='compare_gwb_gmr_gpf_tau3', save=False)

# ---- GMR TAU=1
gwb_amps(sams=sams_list, dpops=None, fname='compare_amps_gmr_tau1', 
         color_hardmods=False, gpf_flags=gpf_flags)

compare_gwb_sim_vs_sam(sams_list, [], gpf_flags=gpf_flags, 
                       fname_extra='compare_gwb_gmr_tau1', save=False)
## ---- GMR TAU=3
#gwb_amps(sams=sams_gmr_tau3, dpops=None, fname='compare_amps_gmr_tau3', color_hardmods=False)

#compare_gwb_sim_vs_sam(sams_gmr_tau3, [], gpf_flags=[0]*len(sams_gmr_tau1), 
#                       fname_extra='compare_gwb_gmr_tau3', save=False)
## ---- GPF TAU=1
#gwb_amps(sams=sams_gpf_tau1, dpops=None, fname='compare_amps_gpf_tau1', color_hardmods=False)

#compare_gwb_sim_vs_sam(sams_gpf_tau1, [], gpf_flags=[1]*len(sams_gpf_tau1), 
#                       fname_extra='compare_gwb_gpf_tau1', save=False)
## ---- GPF TAU=3
#gwb_amps(sams=sams_gpf_tau3, dpops=None, fname='compare_amps_gpf_tau3', color_hardmods=False)

#compare_gwb_sim_vs_sam(sams_gpf_tau3, [], gpf_flags=[1]*len(sams_gpf_tau1), 
#                       fname_extra='compare_gwb_gpf_tau3', save=False)


In [ ]:
# --- compare GWB amps for TNG50, TNG100, and Illustris using strict Npart cuts and mmod
lbl_list = ['fsa-mm-Illustris-1','fsa-mm-TNG50-1', 'fsa-mm-TNG100-1', 'fsa-mm-TNG300-1']
#sim_hardmod_list = ['old_rc10', 'old_rc100', 'old_rc10_nu0', 'ph15', 'ph15_rc10']
sim_hardmod_list = ['old_rc100','ph15_rc10']
sim_tau_list = [1.0]
#sim_tau_list = [3.0] ### havent run the tau=3.0 sims yet
dp_list = [ d for d in ill_dpops_mmod+t50_dpops_mmod+t100_dpops_mmod+t300_dpops_mmod 
            if (d.lbl in lbl_list and d.hard_model_type in sim_hardmod_list 
                and d.HARD_PARS['hard_time'] in sim_tau_list) ]

#sam_mod_list =['old','old_2s','old_rc100','ph15','astr', 'astr_nuo0','astr_rc100']
#sam_mod_list = ['ph15','astr_rc100','astr'] # high, med, low amps
sam_mod_list = ['old_rc100','astr'] # high, med, low amps
gpfflag_list = [0]  # [0, 1]
#gpfflag_list = [1]  # [0, 1]
sam_tau_list = [1.0]
#sam_tau_list = [3.0]
sams_list = [ s for s in sams_gmr_tau1 if (s.model_type in sam_mod_list and
#sams_list = [ s for s in sams_gpf_tau1 if (s.model_type in sam_mod_list and
              s.gpf_flag in gpfflag_list and s.PARS['hard_time'] in sam_tau_list) ]

gwb_amps(sams=sams_list, dpops=dp_list, color_hardmods=False, 
         fname='compare_amps_sim_sam_gmr_tau1')
         #fname='compare_amps_sim_sam_gpf_tau1')

##compare_gwb_sim_vs_sam(freqs_gmr, [gwb_new_sam_gmr,gwb_new_sam_gpf], dp_list, gpf_flags=[0,1], 
##                       fname_extra='gwb_compare')
compare_gwb_sim_vs_sam(sams_list, dp_list, gpf_flags=[], save=False,
                       fname_extra='compare_gwb_sim_sam_gmr_tau1')
                       #fname_extra='compare_gwb_sim_sam_gpf_tau1')

#compare_bin_pops(dp_list[::len(sim_hardmod_list)], fname_extra='sim_compare_mmod', colors=['g','r','b','m'])

# --- old model comparison
#print(sams_gmr_tau1[0].model_type, dp_list[0].hard_model_type)
#plot_loud_binary_params(sams_gmr_tau1[0], dp_list[0], gpf_flag=0, save=False)

#print(sams_gmr_tau1[0].model_type, dp_list[2].hard_model_type)
#plot_loud_binary_params(sams_gmr_tau1[0], dp_list[2], gpf_flag=0, save=False)

#print(sams_gmr_tau1[0].model_type, dp_list[4].hard_model_type)
#plot_loud_binary_params(sams_gmr_tau1[0], dp_list[4], gpf_flag=0, save=False)

#print(sams_gmr_tau1[0].model_type, dp_list[6].hard_model_type)
#plot_loud_binary_params(sams_gmr_tau1[0], dp_list[6], gpf_flag=0, save=False)

# --- old model comparison but with consistent rc=100 for sims and sams
#print(sams_gmr_tau1[2].model_type, dp_list[0].hard_model_type)
#plot_loud_binary_params(sams_gmr_tau1[2], dp_list[0], gpf_flag=0, save=False)

#print(sams_gmr_tau1[2].model_type, dp_list[2].hard_model_type)
#plot_loud_binary_params(sams_gmr_tau1[2], dp_list[2], gpf_flag=0, save=False)

#print(sams_gmr_tau1[2].model_type, dp_list[4].hard_model_type)
#plot_loud_binary_params(sams_gmr_tau1[2], dp_list[4], gpf_flag=0, save=False)

#print(sams_gmr_tau1[2].model_type, dp_list[6].hard_model_type)
#plot_loud_binary_params(sams_gmr_tau1[2], dp_list[6], gpf_flag=0, save=False)

# --- astr model comparison
#print(sams_gmr_tau1[4].model_type, dp_list[1].hard_model_type)
#plot_loud_binary_params(sams_gmr_tau1[4], dp_list[1], gpf_flag=0, save=False)

#print(sams_gmr_tau1[4].model_type, dp_list[3].hard_model_type)
#plot_loud_binary_params(sams_gmr_tau1[4], dp_list[3], gpf_flag=0, save=False)

#print(sams_gmr_tau1[4].model_type, dp_list[5].hard_model_type)
#plot_loud_binary_params(sams_gmr_tau1[4], dp_list[5], gpf_flag=0, save=False)

#print(sams_gmr_tau1[4].model_type, dp_list[7].hard_model_type)
#plot_loud_binary_params(sams_gmr_tau1[4], dp_list[7], gpf_flag=0, save=False)


In [ ]:
# --- compare GWB amps for different Illustris Npart cuts
lbl_list = ['fsa-mm-Illustris-1-N001-bh1',
            'fsa-mm-Illustris-1-N010-bh0',
            'fsa-mm-Illustris-1-N010-bh1',
            'fsa-mm-Illustris-1-bh0',
            'fsa-mm-Illustris-1']
dp_list = [d for d in ill_dpops_mmod if d.lbl in lbl_list]


#sam_mod_list = ['ph15','astr_rc100']
#sam_mod_list =['old','old_2s','old_rc100','ph15','astr', 'astr_nuo0','astr_rc100']
#sam_mod_list = ['ph15','astr_rc100','astr'] # high, med, low amps
#sam_tau_list = [1.0, 3.0]
sam_tau_list = [1.0]
#sam_tau_list = [3.0]
gmr_sam_mod_list = ['astr_rc100']
gpf_sam_mod_list = ['ph15']
gmr_sams_list = [ s for s in sams_gmr_tau1+sams_gmr_tau3
                  if (s.model_type in gmr_sam_mod_list and 
                  s.PARS['hard_time'] in sam_tau_list) ]
gpf_sams_list = [ s for s in sams_gpf_tau1+sams_gpf_tau3
                  if (s.model_type in gpf_sam_mod_list and 
                  s.PARS['hard_time'] in sam_tau_list) ]

sams_list = gmr_sams_list + gpf_sams_list
gpf_flags = [0]*len(gmr_sams_list) + [1]*len(gpf_sams_list) 


gwb_amps(sams=sams_list, dpops=dp_list, color_hardmods=False, gpf_flags=gpf_flags,
         fname='compare_amps_Illustris_SAM_tau1')
         #fname='compare_amps_sim_sam_gpf_tau1')

compare_gwb_sim_vs_sam(sams_list, dp_list, gpf_flags=gpf_flags, save=False,
                       fname_extra='compare_gwb_Illustris_SAM_tau1')
                       #fname_extra='compare_gwb_sim_sam_gpf_tau1')


for d in dp_list:
    print(gmr_sams_list[0].model_type, d.hard_model_type)
    plot_loud_binary_params(gmr_sams_list[0], d, gpf_flag=0, save=False)


In [ ]:
# --- compare GWB amps for TNG100 only, using different Npart cuts and resolution
lbl_list = ['fsa-mm-TNG100-1-N010-bh0',
            'fsa-mm-TNG100-1-bh0', 'fsa-mm-TNG100-1', 'fsa-mm-TNG100-2']
dp_list = [d for d in tng100_dpops_mmod if d.lbl in lbl_list]

gwb_amps(freqs_gpf, freqs_gmr, gwb_sam_gpf_hctot, gwb_sam_gmr_hctot, 
         fname='compare_gwb_amps_TNG100', dpops=dp_list)

compare_gwb_sim_vs_sam(freqs_gmr, [gwb_new_sam_gmr,gwb_new_sam_gpf], dp_list, gpf_flags=[0,1], 
                       fname_extra='gwb_compare_TNG100')

compare_bin_pops(dp_list, fname_extra='TNG100', colors=['g','c','darkgreen','b','k'])

for dp in [dp_list[0],dp_list[1],dp_list[-1]]:
    plot_loud_binary_params(freqs_gmr, gwb_sam_gmr_bgpars, gwb_sam_gmr_sspars, 
                            dp, gpf_flag=0, save=True)

In [ ]:
# --- compare GWB amps for TNG50-1 only, using different Npart cuts
lbl_list = ['fsa-mm-TNG50-1-N100-bh0', 'fsa-mm-TNG50-1-N100',
            'fsa-mm-TNG50-1-bh0', 'fsa-mm-TNG50-1'] 
#            'fsa-mm-TNG50-2', 'fsa-mm-TNG50-3']
dp_list = [d for d in tng50_dpops_mmod if d.lbl in lbl_list]

gwb_amps(freqs_gpf, freqs_gmr, gwb_sam_gpf_hctot, gwb_sam_gmr_hctot, 
         fname='compare_gwb_amps_TNG50-1', dpops=dp_list)

compare_gwb_sim_vs_sam(freqs_gmr, [gwb_new_sam_gmr,gwb_new_sam_gpf], dp_list, gpf_flags=[0,1], 
                       fname_extra='gwb_compare_TNG50-1')

compare_bin_pops(dp_list, fname_extra='TNG50-1', colors=['darkred','darkred','r','r'])

for dp in dp_list[:-1]:
    plot_loud_binary_params(freqs_gmr, gwb_sam_gmr_bgpars, gwb_sam_gmr_sspars, 
                            dp, gpf_flag=0, save=True)

In [ ]:
# --- compare GWB amps for TNG50, TNG100, and Illustris using strict Npart cuts and different res
lbl_list = ['fsa-mm-Illustris-1','fsa-mm-TNG300-1','fsa-mm-TNG100-1','fsa-mm-TNG100-2', 
            'fsa-mm-TNG50-1','fsa-mm-TNG50-2','fsa-mm-TNG50-3']
dp_list = [d for d in ill_dpops_mmod+tng300_dpops_mmod+tng100_dpops_mmod+tng50_dpops_mmod
           if d.lbl in lbl_list]

gwb_amps(freqs_gpf, freqs_gmr, gwb_sam_gpf_hctot, gwb_sam_gmr_hctot, 
         fname='compare_gwb_amps_vary_res', dpops=dp_list)

compare_gwb_sim_vs_sam(freqs_gmr, [gwb_new_sam_gmr,gwb_new_sam_gpf], dp_list, gpf_flags=[0,1], 
                       fname_extra='gwb_compare_vary_res')

compare_bin_pops(dp_list, fname_extra='vary_res', colors=['g', 'b','c','r', 'darkorange','y'])

for dp in [dp_list[1],dp_list[-2],dp_list[-1]]:
    plot_loud_binary_params(freqs_gmr, gwb_sam_gmr_bgpars, gwb_sam_gmr_sspars, 
                            dp, gpf_flag=0, save=True)

In [ ]:
## ---- Load 'old' GPF SAM
#tmp_old_gpf = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, model_type='old', gpf_flag=True)
##sam_gpf, hard_gpf, gwb_new_sam_gpf, gwb_sam_gpf, freqs_gpf, freqs_edges_gpf = tmp_gpf
#sam_old_gpf, hard_old_gpf, gwb_sam_old_gpf, MODEL_PARS_old_gpf = tmp_old_gpf
#print(f"{MODEL_PARS_old_gpf['freqs'].shape=}")
#gwb_sam_old_gpf_hctot = np.sqrt( np.sum(gwb_sam_old_gpf[0]**2,axis=2) + gwb_sam_old_gpf[1]**2 )

##gwb_sam_gpf_hcss = gwb_sam_gpf[0]    # shape: [nfreqs, nreals, nloudest]
##gwb_sam_gpf_hcbg = gwb_sam_gpf[1]    # shape: [nfreqs, nreals]
##gwb_sam_gpf_hctot = np.sqrt( np.sum(gwb_sam_gpf[0]**2,axis=2) + gwb_sam_gpf[1]**2 )
##gwb_sam_gpf_sspars = gwb_sam_gpf[2]    # shape: [nsspars, nfreqs, nreals, nloudest]
##gwb_sam_gpf_bgpars = gwb_sam_gpf[3]    # shape: [nfreqs, nreals]

## ---- Load 'old' GMR SAM
#tmp_old_gmr = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, model_type='old', gpf_flag=False)
##sam_gmr, hard_gmr, gwb_new_sam_gmr, gwb_sam_gmr, freqs_gmr, freqs_edges_gmr = tmp_gmr
#sam_old_gmr, hard_old_gmr, gwb_sam_old_gmr, MODEL_PARS_old_gmr = tmp_old_gmr
#gwb_sam_old_gmr_hctot = np.sqrt( np.sum(gwb_sam_old_gmr[0]**2,axis=2) + gwb_sam_old_gmr[1]**2 )

##gwb_sam_gmr_hcss = gwb_sam_gmr[0]    # shape: [nfreqs, nreals, nloudest]
##gwb_sam_gmr_hcbg = gwb_sam_gmr[1]    # shape: [nfreqs, nreals]
##gwb_sam_gmr_hctot = np.sqrt( np.sum(gwb_sam_gmr[0]**2,axis=2) + gwb_sam_gmr[1]**2 )
##gwb_sam_gmr_sspars = gwb_sam_gmr[2]    # shape: [nsspars, nfreqs, nreals, nloudest]
##gwb_sam_gmr_bgpars = gwb_sam_gmr[3]    # shape: [nfreqs, nreals]

## ---- load other model_type SAMs:
#tmp_old2s_gpf = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, model_type='old_with_dblSchechter', gpf_flag=True)
#sam_old2s_gpf, hard_old2s_gpf, gwb_sam_old2s_gpf, MODEL_PARS_old2s_gpf = tmp_old2s_gpf
#gwb_sam_old2s_gpf_hctot = np.sqrt( np.sum(gwb_sam_old2s_gpf[0]**2,axis=2) + gwb_sam_old2s_gpf[1]**2 )

#tmp_old2s_gmr = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, model_type='old_with_dblSchechter', gpf_flag=False)
#sam_old2s_gmr, hard_old2s_gmr, gwb_sam_old2s_gmr, MODEL_PARS_old2s_gmr = tmp_old2s_gmr
#gwb_sam_old2s_gmr_hctot = np.sqrt( np.sum(gwb_sam_old2s_gmr[0]**2,axis=2) + gwb_sam_old2s_gmr[1]**2 )

#tmp_ph15_gpf = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, model_type='phenom15yr', gpf_flag=True)
#sam_ph15_gpf, hard_ph15_gpf, gwb_sam_ph15_gpf, MODEL_PARS_ph15_gpf = tmp_ph15_gpf
#gwb_sam_ph15_gpf_hctot = np.sqrt( np.sum(gwb_sam_ph15_gpf[0]**2,axis=2) + gwb_sam_ph15_gpf[1]**2 )

#tmp_ph15_gmr = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, model_type='phenom15yr', gpf_flag=False)
#sam_ph15_gmr, hard_ph15_gmr, gwb_sam_ph15_gmr, MODEL_PARS_ph15_gmr = tmp_ph15_gmr
#gwb_sam_ph15_gmr_hctot = np.sqrt( np.sum(gwb_sam_ph15_gmr[0]**2,axis=2) + gwb_sam_ph15_gmr[1]**2 )

#tmp_astr_gpf = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, model_type='astrostrong', gpf_flag=True)
#sam_astr_gpf, hard_astr_gpf, gwb_sam_astr_gpf, MODEL_PARS_astr_gpf = tmp_astr_gpf
#gwb_sam_astr_gpf_hctot = np.sqrt( np.sum(gwb_sam_astr_gpf[0]**2,axis=2) + gwb_sam_astr_gpf[1]**2 )

#tmp_astr_gmr = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, model_type='astrostrong', gpf_flag=False)
#sam_astr_gmr, hard_astr_gmr, gwb_sam_astr_gmr, MODEL_PARS_astr_gmr = tmp_astr_gmr
#gwb_sam_astr_gmr_hctot = np.sqrt( np.sum(gwb_sam_astr_gmr[0]**2,axis=2) + gwb_sam_astr_gmr[1]**2 )
